# Week 6 – Feature Engineering

This is a standalone notebook for Week 6. Run the cells from top to bottom.

**Input:** `../Dataset/youth_drug_consumption_cleaned.csv` (generated in Week 3).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
from pathlib import Path

cleaned_path = Path("../Dataset/youth_drug_consumption_cleaned.csv")
if not cleaned_path.exists():
    raise FileNotFoundError("Cleaned dataset not found. Run Week 3 first and save youth_drug_consumption_cleaned.csv in the Dataset folder.")
df_clean = pd.read_csv(cleaned_path)
print("Cleaned Dataset Shape:", df_clean.shape)
display(df_clean.head())

# WEEK 6 – FEATURE ENGINEERING

## Aim
To transform the cleaned dataset using suitable preprocessing and feature engineering techniques.

## Activities Performed
1. Numerical and categorical feature identification.
2. Label Encoding.
3. One-Hot Encoding.
4. Standardization.
5. Min-Max Scaling.
6. Skewness analysis.
7. Yeo-Johnson transformation.
8. Binning and discretization.
9. Before-and-after comparison.

## Outcome
Different data transformation techniques were applied to prepare the dataset for further analysis.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create a separate copy
df_fe = df_clean.copy()

print("Dataset Shape:", df_fe.shape)
df_fe.head()

## 6.1 Identification of Numerical and Categorical Features

Features are identified based on their data types and meanings.

Numerical features contain measurable values, while categorical features represent groups or categories.

In [ ]:
personality_cols = [
    "Nscore", "Escore", "Oscore",
    "Ascore", "Cscore", "Impulsive", "SS"
]

drug_cols = [
    "Alcohol", "Amphet", "Amyl", "Benzos",
    "Caff", "Cannabis", "Choc", "Coke",
    "Crack", "Ecstasy", "Heroin", "Ketamine",
    "Legalh", "LSD", "Meth", "Mushrooms",
    "Nicotine", "Semer", "VSA"
]

print("Numerical Features:")
print(personality_cols)

print("\nCategorical Features:")
print(drug_cols)

## 6.2 Label Encoding

Label Encoding converts categorical values into numerical representations.

The drug consumption categories CL0 to CL6 are converted into numerical values while preserving their order.

In [ ]:
encoding_map = {
    "CL0": 0,
    "CL1": 1,
    "CL2": 2,
    "CL3": 3,
    "CL4": 4,
    "CL5": 5,
    "CL6": 6
}

df_label_encoded = df_fe.copy()

for column in drug_cols:
    df_label_encoded[column] = df_label_encoded[column].map(encoding_map)

print("Label Encoding Completed!")
display(df_label_encoded[drug_cols].head())

## 6.3 One-Hot Encoding

One-Hot Encoding converts categorical values into separate binary columns.

It is useful for categorical variables that do not have a natural numerical order.

In [ ]:
gender_data = df_fe.drop(columns=["Gender"]).copy()

gender_data["Gender_Label"] = df_fe["Gender"].map({
    -0.48246: "Male",
     0.48246: "Female"
})

df_onehot = pd.get_dummies(
    gender_data,
    columns=["Gender_Label"],
    dtype=int
)

print("One-Hot Encoding Completed!")
display(df_onehot.head())

print("New Shape:", df_onehot.shape)

## 6.4 Standardization

Standardization transforms numerical features so that they have approximately zero mean and unit standard deviation.

StandardScaler is used for the personality score variables.

In [ ]:
scaler = StandardScaler()

df_standard = df_fe.copy()

df_standard[personality_cols] = scaler.fit_transform(
    df_fe[personality_cols]
)

print("Standardization Completed!")
display(df_standard[personality_cols].head())

print("\nMean after standardization:")
print(df_standard[personality_cols].mean().round(2))

print("\nStandard deviation:")
print(df_standard[personality_cols].std().round(2))

In [ ]:
minmax_scaler = MinMaxScaler()

df_minmax = df_fe.copy()

df_minmax[personality_cols] = minmax_scaler.fit_transform(
    df_fe[personality_cols]
)

print("Min-Max Scaling Completed!")
display(df_minmax[personality_cols].head())

## 6.5 Skewness Analysis

Skewness measures the asymmetry of a data distribution.

- Positive skewness indicates a longer right tail.
- Negative skewness indicates a longer left tail.
- Skewness near zero indicates a relatively symmetrical distribution.

In [ ]:
skew_values = df_fe[personality_cols].skew()

print("Skewness Values:")
print(skew_values)

plt.figure(figsize=(10, 5))

sns.barplot(
    x=skew_values.index,
    y=skew_values.values
)

plt.axhline(0, linestyle="--", color="black")
plt.title("Skewness of Personality Features")
plt.xlabel("Features")
plt.ylabel("Skewness")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6.6 Skewness Correction

Yeo-Johnson transformation is used to reduce skewness in numerical data.

It can handle both positive and negative values.
Features with absolute skewness greater than 0.5 are selected for transformation.

In [ ]:
skewed_cols = skew_values[
    abs(skew_values) > 0.5
].index.tolist()

print("Selected Skewed Features:")
print(skewed_cols)

df_transformed = df_fe.copy()

if skewed_cols:
    transformer = PowerTransformer(
        method="yeo-johnson",
        standardize=False
    )

    df_transformed[skewed_cols] = transformer.fit_transform(
        df_fe[skewed_cols]
    )

    print("Yeo-Johnson Transformation Completed!")

else:
    print("No features selected for transformation.")

display(df_transformed[personality_cols].head())

## 6.7 Before and After Transformation Comparison

The skewness values before and after Yeo-Johnson transformation are compared to understand the effect of the transformation.

In [ ]:
before_skew = df_fe[personality_cols].skew()
after_skew = df_transformed[personality_cols].skew()

skew_comparison = pd.DataFrame({
    "Before Transformation": before_skew,
    "After Transformation": after_skew
})

display(skew_comparison)

skew_comparison.plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Skewness Before and After Transformation")
plt.xlabel("Personality Features")
plt.ylabel("Skewness")
plt.axhline(0, color="black", linestyle="--")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6.8 Binning and Discretization

Binning is the process of dividing continuous numerical data into intervals or groups.

Two methods are used:

1. Equal-width binning – divides the data into intervals of equal width.
2. Equal-frequency binning – divides the data into groups containing approximately equal numbers of observations.

In [ ]:
# Equal-width binning
df_binned = df_fe.copy()

df_binned["Nscore_EqualWidth"] = pd.cut(
    df_binned["Nscore"],
    bins=4,
    labels=["Low", "Medium", "High", "Very High"]
)

# Equal-frequency binning
df_binned["Nscore_EqualFrequency"] = pd.qcut(
    df_binned["Nscore"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

print("Binning Completed!")

display(
    df_binned[
        ["Nscore", "Nscore_EqualWidth", "Nscore_EqualFrequency"]
    ].head(10)
)

## 6.9 Visualization of Binning

The distribution of personality score categories is visualized using count plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(
    data=df_binned,
    x="Nscore_EqualWidth",
    ax=axes[0]
)

axes[0].set_title("Equal-Width Binning")

sns.countplot(
    data=df_binned,
    x="Nscore_EqualFrequency",
    ax=axes[1]
)

axes[1].set_title("Equal-Frequency Binning")

plt.tight_layout()
plt.show()

## 6.10 Experiment Conclusion

Data transformation techniques were applied to the drug consumption dataset.

Label Encoding and One-Hot Encoding were used for categorical features. Standardization and Min-Max Scaling were used for feature scaling. Skewness was analyzed and Yeo-Johnson transformation was applied to selected features. Binning and discretization were performed using equal-width and equal-frequency methods.

These techniques helped understand and transform the dataset for further analysis.

In [ ]:
print("EXPERIMENT 5 - FINAL SUMMARY")

print("\nOriginal Dataset:", df_clean.shape)
print("Label Encoded Dataset:", df_label_encoded.shape)
print("One-Hot Encoded Dataset:", df_onehot.shape)
print("Standardized Dataset:", df_standard.shape)
print("Transformed Dataset:", df_transformed.shape)
print("Binned Dataset:", df_binned.shape)

print("\nExperiment 5 Completed Successfully!")